# Carrinho IESB — A* e explicação pelo Qwen, passo a passo

**Primeira etapa da integração dos Laboratórios 2 e 7.**
Fluxo: nomes cadastrados → A* → dados calculados → explicação em português.
Usamos o **mapa antigo**, campus provisório de 50 × 30 m. O croqui medido
ainda não foi aplicado. As distâncias deste cenário não são medições reais.

Abra este arquivo no Colab e execute as células de cima para baixo.
Para a parte do Qwen, selecione uma GPU em **Ambiente de execução →
Alterar tipo de ambiente de execução**, antes de começar. Em CPU é mais lento.
O modelo é baixado da Hugging Face e ocupa alguns GB; essa parte usa internet.
A parte de A* funciona sem GPU, sem baixar modelos e sem abrir o Webots.

Cada bloco apresenta a ideia antes do código. Para mudar o exemplo,
altere apenas `origem` e `destino` na seção 5 e execute as células seguintes.
Reiniciar o ambiente apaga as variáveis: nesse caso, comece novamente do topo.

## 0. Disponibilizar a base que já existe

A célula abaixo contém uma cópia dos cinco arquivos Python necessários.
Ela os escreve em uma pasta temporária e permite importá-los. Não extrai
backups nem modifica os arquivos do projeto. O código do A* continua em
`buscas.py`; a geometria continua em `campus_config.py`.
Esta preparação está recolhida no Colab por ser longa; o conteúdo de estudo
começa na seção 1. É possível expandi-la para consultar as cópias dos arquivos.

In [ ]:
#@title Preparar os cinco arquivos do mapa antigo (execute uma vez)
# pathlib trabalha com caminhos; tempfile cria uma pasta temporária nova.
# sys.path é a lista de pastas em que o Python procura módulos para importar.
from pathlib import Path
import tempfile
import sys

# Este dicionário contém o texto dos cinco arquivos, incorporado pelo gerador.
arquivos_do_mapa = {'campus_config.py': '"""Etapa 2 — esquema PROVISÓRIO, não é uma planta oficial do IESB.\n\nTodas as medidas em metros. X: direita no mapa; Y: interior do campus;\nZ: altura. Origem (0, 0, 0) na Entrada IESB, sobre o piso.\nApós editar: python gerar_campus.py; depois recarregue o mundo no Webots.\n"""\n\nimport math\n\n# Limites do piso; alterar não redimensiona automaticamente os outros elementos.\npiso = {"x_min": -25.0, "x_max": 25.0, "y_min": -2.0, "y_max": 28.0,\n        "espessura": 0.10}\nparedes_padrao = {"espessura": 0.16, "altura": 0.70}\nentrada = {"centro_x": 0.0, "largura": 4.0}  # Abertura na borda y_min.\n\n# Pontos de referência: identificador -> (X, Y), em metros.\n# Referências de locais; grade_navegacao.py associa cada uma a uma célula.\npontos = {\n    "EXT01": (0.0, 0.0),\n    "EXT02": (0.0, 12.5),\n    "EXT03": (-4.5, 4.5),\n    "EXT04": (4.5, 4.5),\n    "EXT05": (-4.5, 12.5),\n    "EXT06": (-12.5, 18.0),\n    "EXT07": (12.5, 18.0),\n    "EXT08": (4.5, 12.5),\n    # Destinos internos: o carrinho cruza a porta e chega ao centro do setor.\n    "BB01": (-10.5, 4.5),\n    "CDA01": (10.5, 4.5),\n    "H01": (4.5, 23.5),\n    "F01": (12.5, 23.5),\n    "J01": (20.5, 23.5),\n    "E01": (-10.5, 12.5),\n    "D01": (-4.5, 23.5),\n    "C01": (-12.5, 23.5),\n    "B01": (-20.5, 23.5),\n    "CAN01": (10.5, 12.5),\n}\n# Ao adicionar um ponto, cadastre o mesmo identificador neste dicionário.\nnomes = {\n    "EXT01": "Entrada IESB",\n    "EXT02": "Cruzamento Central",\n    "EXT03": "Acesso Biblioteca",\n    "EXT04": "Acesso Central de Atendimento",\n    "EXT05": "Acesso Bloco E",\n    "EXT06": "Acesso Blocos B/C/D",\n    "EXT07": "Acesso Blocos H/F/J",\n    "EXT08": "Acesso Cantina",\n    "BB01": "Biblioteca", "CDA01": "Central de Atendimento",\n    "H01": "Bloco H", "F01": "Bloco F", "J01": "Bloco J",\n    "E01": "Bloco E", "D01": "Bloco D", "C01": "Bloco C",\n    "B01": "Bloco B", "CAN01": "Cantina",\n}\n\n# Cada setor é um retângulo aberto em cima, com quatro paredes e uma porta.\n# centro = (x, y); tamanho = (largura X, comprimento Y).\n# porta: lado N(+Y), S(-Y), L(+X) ou O(-X), largura e deslocamento\n# em relação ao meio da parede (positivo em X para N/S, em Y para L/O).\n# Ampliação de 21/10/2026: centros e eixos das portas alinhados aos centros da\n# grade (coordenadas terminadas em .5). Corredores preservados ao afastar setores.\nsetores = [\n    {"id": "BIBLIOTECA", "nome": "Biblioteca", "centro": pontos["BB01"],\n     "tamanho": (8.0, 6.0), "porta": ("L", 2.4, 0.0)},\n    {"id": "ATENDIMENTO", "nome": "Central de Atendimento", "centro": pontos["CDA01"],\n     "tamanho": (8.0, 6.0), "porta": ("O", 2.4, 0.0)},\n    {"id": "BLOCO_E", "nome": "Bloco E", "centro": pontos["E01"],\n     "tamanho": (8.0, 6.0), "porta": ("L", 2.4, 0.0)},\n    {"id": "CANTINA", "nome": "Cantina", "centro": pontos["CAN01"],\n     "tamanho": (8.0, 6.0), "porta": ("O", 2.4, 0.0)},\n    {"id": "BLOCO_B", "nome": "Bloco B", "centro": pontos["B01"],\n     "tamanho": (6.0, 6.0), "porta": ("S", 2.4, 0.0)},\n    {"id": "BLOCO_C", "nome": "Bloco C", "centro": pontos["C01"],\n     "tamanho": (6.0, 6.0), "porta": ("S", 2.4, 0.0)},\n    {"id": "BLOCO_D", "nome": "Bloco D", "centro": pontos["D01"],\n     "tamanho": (6.0, 6.0), "porta": ("S", 2.4, 0.0)},\n    {"id": "BLOCO_H", "nome": "Bloco H", "centro": pontos["H01"],\n     "tamanho": (6.0, 6.0), "porta": ("S", 2.4, 0.0)},\n    {"id": "BLOCO_F", "nome": "Bloco F", "centro": pontos["F01"],\n     "tamanho": (6.0, 6.0), "porta": ("S", 2.4, 0.0)},\n    {"id": "BLOCO_J", "nome": "Bloco J", "centro": pontos["J01"],\n     "tamanho": (6.0, 6.0), "porta": ("S", 2.4, 0.0)},\n]\ndestinos_setores = {"BIBLIOTECA": "BB01", "ATENDIMENTO": "CDA01", "CANTINA": "CAN01",\n                    **{f"BLOCO_{letra}": f"{letra}01" for letra in "BCDEHFJ"}}\n\n# Paredes adicionais: centro (x,y), tamanho (X,Y,Z), alinhadas aos eixos.\nparedes = []\n# Blocos sólidos simples para representar mobiliário, sem reconhecimento.\nblocos = [\n    {"id": "BANCO_OESTE", "centro": (-18.0, 17.5), "tamanho": (2.0, 0.6, 0.45)},\n    {"id": "BANCO_LESTE", "centro": (18.0, 17.5), "tamanho": (2.0, 0.6, 0.45)},\n]\n\n# Escala 3: chassi de 0,72 x 0,48 m; largura com rodas de 0,675 m.\nrobo = {"ponto_inicial": "EXT01", "angulo": math.pi / 2, "escala": 3.0}\nmarcadores = {"raio": 0.20, "altura": 0.004,\n              "tamanho_rotulo": (1.45, 0.42), "deslocamento_rotulo": (0.0, 0.55)}\n# Enquadramento da vista superior; não é uma câmera instalada no robô.\nvista = {"campo_visual": 0.785398, "proporcao_janela": 2.2, "margem": 1.1}\n# Margem além do raio que engloba chassi e rodas. A escala é aplicada pelo gerador.\nfolga_robo = 0.05\n\n# Células alinhadas às linhas existentes; quatro vizinhos, sem diagonais.\n# A associação de um local a um centro exige segmento livre e distância limitada.\ngrade = {"passo": 1.0, "distancia_max_associacao": 1.5}\n', 'gerar_campus.py': '"""Gera os mundos da etapa 2 a partir de campus_config.py; não controla o robô."""\n\nimport argparse\nimport json\nimport math\nfrom pathlib import Path\nimport re\n\nimport campus_config as cfg\n\nRAIZ = Path(__file__).resolve().parent\n\n\ndef caixas_cenario(config=cfg):\n    """Retângulos de colisão das paredes e blocos, também usados na validação."""\n    caixas = []\n    t, h = config.paredes_padrao["espessura"], config.paredes_padrao["altura"]\n\n    def caixa(ident, centro, tamanho):\n        caixas.append({"id": ident, "centro": centro, "tamanho": tamanho})\n\n    def parede(ident, horizontal, fixo, inicio, fim):\n        if fim <= inicio:\n            raise ValueError(f"Parede {ident}: comprimento deve ser positivo")\n        centro = ((inicio + fim) / 2, fixo) if horizontal else (fixo, (inicio + fim) / 2)\n        tamanho = (fim - inicio, t, h) if horizontal else (t, fim - inicio, h)\n        caixa(ident, centro, tamanho)\n\n    p = config.piso\n    xa, xb, ya, yb = (p[k] for k in ("x_min", "x_max", "y_min", "y_max"))\n    entrada_x, entrada_l = config.entrada["centro_x"], config.entrada["largura"]\n    if not (0 < entrada_l and xa < entrada_x - entrada_l/2 < entrada_x + entrada_l/2 < xb):\n        raise ValueError("A entrada deve caber na borda do piso")\n    lado_entrada = config.entrada.get("lado", "S")\n    if lado_entrada not in ("N", "S"):\n        raise ValueError("Lado da entrada deve ser N ou S")\n    fechado, aberto = (yb-t/2, ya+t/2) if lado_entrada == "S" else (ya+t/2, yb-t/2)\n    parede("LIMITE_"+("N" if lado_entrada == "S" else "S"), True, fechado, xa, xb)\n    parede("LIMITE_O", False, xa+t/2, ya, yb)\n    parede("LIMITE_L", False, xb-t/2, ya, yb)\n    parede(f"LIMITE_{lado_entrada}_O", True, aberto, xa, entrada_x-entrada_l/2)\n    parede(f"LIMITE_{lado_entrada}_L", True, aberto, entrada_x+entrada_l/2, xb)\n    for s in config.setores:\n        x, y = s["centro"]\n        w, d = s["tamanho"]\n        if s.get("porta") is None:\n            # Volume sem interior mapeado: não inventa porta nem acesso transitável.\n            caixa(s["id"]+"_VOLUME", (x, y), (w, d, s.get("altura", h)))\n            continue\n        lado_porta, largura, offset = s["porta"]\n        if lado_porta not in ("N", "S", "L", "O") or min(w, d) <= 2*t:\n            raise ValueError(f"Setor inválido: {s[\'id\']}")\n        comprimento = w if lado_porta in ("N", "S") else d\n        if not (0 < largura and abs(offset) + largura/2 < comprimento/2-t):\n            raise ValueError(f"Porta não cabe no setor {s[\'id\']}")\n        for lado, horizontal, fixo, inicio, fim, meio in [\n            ("N", True, y+d/2, x-w/2, x+w/2, x),\n            ("S", True, y-d/2, x-w/2, x+w/2, x),\n            ("L", False, x+w/2, y-d/2, y+d/2, y),\n            ("O", False, x-w/2, y-d/2, y+d/2, y),\n        ]:\n            ident = f"{s[\'id\']}_{lado}"\n            if lado == lado_porta:\n                parede(ident+"_1", horizontal, fixo, inicio, meio+offset-largura/2)\n                parede(ident+"_2", horizontal, fixo, meio+offset+largura/2, fim)\n            else:\n                parede(ident, horizontal, fixo, inicio, fim)\n    caixas.extend(config.paredes)\n    caixas.extend(config.blocos)\n    return caixas\n\n\ndef restricoes_circulacao(config=cfg):\n    """Complemento exato da união de faixas mapeadas; não são paredes físicas.\n\n    Particiona nas bordas dos retângulos e une intervalos adjacentes. As caixas\n    resultantes impedem atalhos por gramados/interiores sem acesso documentado.\n    A grade e o seguidor expandem estas caixas pelas mesmas margens do robô.\n    """\n    faixas = getattr(config, "circulacao", None)\n    if faixas is None:\n        return []  # Compatibilidade com configurações de ensaio anteriores.\n    if not faixas:\n        raise ValueError("Circulação mapeada não pode ser vazia")\n    p = config.piso\n    xs, ys = {p["x_min"], p["x_max"]}, {p["y_min"], p["y_max"]}\n    limites = []\n    for faixa in faixas:\n        xa, xb, ya, yb = faixa["limites"]\n        if not all(math.isfinite(v) for v in (xa, xb, ya, yb)) or not (\n                p["x_min"] <= xa < xb <= p["x_max"] and\n                p["y_min"] <= ya < yb <= p["y_max"]):\n            raise ValueError("Faixa de circulação inválida: " + faixa["id"])\n        xs.update((xa, xb)); ys.update((ya, yb))\n        limites.append((xa, xb, ya, yb))\n    xs, ys = sorted(xs), sorted(ys)\n    retangulos, ativos = [], {}\n    for ya, yb in zip(ys, ys[1:]):\n        runs, inicio = [], None\n        for xa, xb in zip(xs, xs[1:]):\n            x, y = (xa+xb)/2, (ya+yb)/2\n            livre = any(a < x < b and c < y < d for a,b,c,d in limites)\n            if not livre and inicio is None:\n                inicio = xa\n            if livre and inicio is not None:\n                runs.append((inicio, xa)); inicio = None\n        if inicio is not None:\n            runs.append((inicio, xs[-1]))\n        novos = {}\n        for xa, xb in runs:\n            chave = (xa, xb)\n            if chave in ativos:\n                rect = ativos[chave]\n                rect[3] = yb\n            else:\n                rect = [xa, xb, ya, yb]\n                retangulos.append(rect)\n            novos[chave] = rect\n        ativos = novos\n    return [{"id": f"SEM_ACESSO_MAPEADO_{i:02d}",\n             "centro": ((a+b)/2, (c+d)/2), "tamanho": (b-a, d-c, .01)}\n            for i, (a,b,c,d) in enumerate(retangulos)]\n\n\ndef validar(config=cfg):\n    """Confere dimensões, identificadores e espaço para o robô em cada ponto."""\n    p = config.piso\n    if not all(math.isfinite(v) for v in p.values()):\n        raise ValueError("Piso deve conter números finitos")\n    if p["x_min"] >= p["x_max"] or p["y_min"] >= p["y_max"] or p["espessura"] <= 0:\n        raise ValueError("Dimensões do piso inválidas")\n    if any(not math.isfinite(v) or v <= 0 for v in config.paredes_padrao.values()):\n        raise ValueError("Altura e espessura das paredes devem ser positivas")\n    if set(config.pontos) != set(config.nomes):\n        raise ValueError("pontos e nomes devem ter os mesmos identificadores")\n    if config.robo["ponto_inicial"] not in config.pontos or not math.isfinite(config.robo["angulo"]):\n        raise ValueError("Posição inicial do robô inválida")\n    escala = config.robo["escala"]\n    if not math.isfinite(escala) or escala <= 0:\n        raise ValueError("Escala do robô deve ser positiva e finita")\n    if not math.isfinite(config.folga_robo) or config.folga_robo < 0:\n        raise ValueError("Folga do robô deve ser finita e não negativa")\n    caixas = caixas_cenario(config)\n    ids = [c["id"] for c in caixas]\n    if len(ids) != len(set(ids)):\n        raise ValueError("Identificadores de paredes/blocos repetidos")\n    for c in caixas:\n        x, y = c["centro"]\n        w, d, h = c["tamanho"]\n        if not re.fullmatch(r"[A-Z][A-Z0-9_]*", c["id"]):\n            raise ValueError(f"Identificador inválido: {c[\'id\']}")\n        if not all(math.isfinite(v) for v in (x,y,w,d,h)) or min(w,d,h) <= 0:\n            raise ValueError(f"Dimensões inválidas: {c[\'id\']}")\n        if not (p["x_min"]-1e-8 <= x-w/2 and x+w/2 <= p["x_max"]+1e-8\n                and p["y_min"]-1e-8 <= y-d/2 and y+d/2 <= p["y_max"]+1e-8):\n            raise ValueError(f"Objeto fora do piso: {c[\'id\']}")\n    # Raio que engloba o carrinho, acrescido da margem de segurança configurada.\n    r = math.hypot(0.12, 0.1125) * escala + config.folga_robo\n    for ident, (x,y) in config.pontos.items():\n        if not re.fullmatch(r"[A-Z][A-Z0-9_]*", ident) or not all(math.isfinite(v) for v in (x,y)):\n            raise ValueError(f"Ponto inválido: {ident}")\n        if not (p["x_min"]+r <= x <= p["x_max"]-r and p["y_min"]+r <= y <= p["y_max"]-r):\n            raise ValueError(f"Ponto fora do piso ou perto demais da borda: {ident}")\n        for c in caixas:\n            cx,cy = c["centro"]\n            w,d,_ = c["tamanho"]\n            if abs(x-cx) <= w/2+r and abs(y-cy) <= d/2+r:\n                raise ValueError(f"Ponto {ident} sem folga para o robô junto de {c[\'id\']}")\n    return caixas\n\n\ndef vetor(valores):\n    """Converte uma sequência de números para o formato do arquivo de mundo."""\n    return " ".join(f"{n:.8g}" for n in valores)\n\n\ndef cor(rgb):\n    """Define uma aparência fosca com a cor vermelha, verde e azul informada."""\n    return f"PBRAppearance {{ baseColor {vetor(rgb)} roughness 1 metalness 0 }}"\n\n\ndef caixa_wbt(ident, centro, tamanho, rgb, colide=True):\n    """Cria o texto de um bloco, com colisão física opcional."""\n    x,y,z = centro\n    return f\'\'\'DEF {ident} Solid {{\n  translation {vetor((x,y,z))}\n  name "{ident}"\n  children [ Shape {{ appearance {cor(rgb)} geometry Box {{ size {vetor(tamanho)} }} }} ]\n  {f\'boundingObject Box {{ size {vetor(tamanho)} }}\' if colide else \'\'}\n}}\n\'\'\'\n\n\ndef textura(ident, texto, fundo):\n    """Salva uma imagem com o nome do setor ou identificador do ponto."""\n    from PIL import Image, ImageDraw, ImageFont\n    img = Image.new("RGB", (1024,256), fundo)\n    draw = ImageDraw.Draw(img)\n    tamanho = 220\n    while True:\n        fonte = ImageFont.load_default(size=tamanho)\n        if draw.textbbox((0,0), texto, font=fonte)[2] < 970 or tamanho <= 20:\n            break\n        tamanho -= 2\n    draw.text((512,128), texto, font=fonte, fill="white", anchor="mm")\n    destino = RAIZ / "worlds" / "textures" / "campus"\n    destino.mkdir(parents=True, exist_ok=True)\n    img.save(destino / f"{ident}.png")\n\n\ndef rotulo(ident, x, y, z, tamanho):\n    """Posiciona a imagem de um rótulo sobre o piso, sem colisão."""\n    return f\'\'\'Pose {{\n  translation {vetor((x,y,z))}\n  children [ Shape {{\n    appearance PBRAppearance {{\n      baseColorMap ImageTexture {{ url ["textures/campus/{ident}.png"] }}\n      roughness 1 metalness 0\n    }}\n    geometry Plane {{ size {vetor(tamanho)} }}\n  }} ]\n}}\n\'\'\'\n\n\ndef gerar():\n    """Gera as texturas e os mundos de controle manual e de teste do campus."""\n    caixas = validar()  # Não sobrescreve os mundos se a configuração for inválida.\n    # Importação local evita ciclo: a grade reutiliza a geometria deste gerador.\n    from grade_navegacao import Grade\n    grade = Grade()\n    p = cfg.piso\n    w,d = p["x_max"]-p["x_min"], p["y_max"]-p["y_min"]\n    cx,cy = (p["x_max"]+p["x_min"])/2, (p["y_max"]+p["y_min"])/2\n    altura_vista = (max(w, d*cfg.vista["proporcao_janela"])\n                    / (2*math.tan(cfg.vista["campo_visual"]/2)) * cfg.vista["margem"])\n    partes = [\'\'\'#VRML_SIM R2025a utf8\n# GERADO por gerar_campus.py. Edite campus_config.py, não este arquivo.\n# Esquema provisório: NÃO é a planta oficial do campus.\nEXTERNPROTO "../protos/Carrinho.proto"\nWorldInfo {\n  title "IESB - campus ampliado - destinos internos"\n  coordinateSystem "ENU"\n  basicTimeStep 32\n  contactProperties [ ContactProperties { material1 "apoio" coulombFriction [ 0.001 ] } ]\n}\n\'\'\', f\'\'\'DEF VISTA Viewpoint {{\n  orientation -0.577350269 0.577350269 0.577350269 2.094395102\n  position {vetor((cx,cy,altura_vista))}\n  fieldOfView {cfg.vista[\'campo_visual\']}\n}}\nBackground {{ skyColor [ 0.75 0.81 0.86 ] }}\nDirectionalLight {{ direction -0.3 0.4 -1 intensity 1 }}\n\'\'\', caixa_wbt("PISO_CAMPUS", (cx,cy,-p["espessura"]/2), (w,d,p["espessura"]), (.79,.81,.80))]\n    # Grade de 1 m puramente visual, sem boundingObject.\n    for x in range(math.ceil(p["x_min"]), math.floor(p["x_max"])+1):\n        partes.append(caixa_wbt(f"GRADE_X_{x-p[\'x_min\']:.0f}", (x,cy,.001), (.012,d,.001), (.67,.70,.70), False))\n    for y in range(math.ceil(p["y_min"]), math.floor(p["y_max"])+1):\n        partes.append(caixa_wbt(f"GRADE_Y_{y-p[\'y_min\']:.0f}", (cx,y,.001), (w,.012,.001), (.67,.70,.70), False))\n    for s in cfg.setores:\n        x,y = s["centro"]\n        sw,sd = s["tamanho"]\n        partes.append(caixa_wbt("AREA_"+s["id"], (x,y,.003), (sw,sd,.002), (.62,.71,.78), False))\n        textura(s["id"], s["nome"], (48,68,89))\n        # O nome fica dentro do bloco, acima do ponto central de chegada.\n        partes.append(rotulo(s["id"],x,y+sd*.30,.008,(sw*.82,.65)))\n    for c in caixas:\n        rgb = (.40,.47,.52) if c in cfg.blocos else (.89,.85,.73)\n        partes.append(caixa_wbt(c["id"], (*c["centro"],c["tamanho"][2]/2), c["tamanho"], rgb))\n    # Células reais de navegação: agrupadas por linha para facilitar a árvore.\n    # Sem boundingObject: a ocupação é uma decisão do planejador, não um objeto físico.\n    componente_entrada = grade.associacoes[cfg.robo["ponto_inicial"]]["componente"]\n    partes.append("DEF GRADE_NAVEGACAO Group { children [")\n    for linha in range(grade.linhas):\n        partes.append(f"DEF LINHA_{linha:02d} Group {{ children [")\n        for celula in (c for c in grade.celulas.values() if c.linha == linha):\n            estado = "LIVRE" if celula.livre else "BLOQUEADA"\n            if celula.livre and grade.componentes[celula.id] != componente_entrada:\n                estado = "ISOLADA"\n            fundo = {"LIVRE": (44, 104, 78), "BLOQUEADA": (132, 58, 60), "ISOLADA": (150, 107, 29)}[estado]\n            curto = f"C{celula.coluna:02d} L{linha:02d}"\n            textura(celula.id, curto, fundo)\n            x, y = celula.centro\n            partes.append(f\'\'\'DEF {celula.id} Solid {{\n  translation {vetor((x,y,.014))}\n  name "{celula.id} - {estado}"\n  children [ {rotulo(celula.id,0,0,0,(.76,.20))} ]\n}}\'\'\')\n        partes.append("] }")\n    partes.append("] }")\n    for ident,(x,y) in cfg.pontos.items():\n        rgb = (.03,.48,.61) if ident.startswith("EXT") else (.80,.31,.08)\n        textura(ident, ident, tuple(round(v*255) for v in rgb))\n        m = cfg.marcadores\n        partes.append(f\'\'\'DEF {ident} Solid {{\n  translation {vetor((x,y,.008))}\n  name {json.dumps(ident+\' - \'+cfg.nomes[ident], ensure_ascii=False)}\n  children [ Shape {{ appearance {cor(rgb)}\n    geometry Cylinder {{ radius {m[\'raio\']} height {m[\'altura\']} }} }} ]\n}}\n\'\'\')\n        ox,oy = m["deslocamento_rotulo"]\n        partes.append(rotulo(ident,x+ox,y+oy,.018,m["tamanho_rotulo"]))\n    for ident,centro,tamanho,rgb in [\n        ("EIXO_X",(1.5,0,.006),(3,.045,.002),(.85,.12,.12)),\n        ("EIXO_Y",(0,1.5,.006),(.045,3,.002),(.08,.55,.20)),\n    ]:\n        partes.append(caixa_wbt(ident,centro,tamanho,rgb,False))\n    for ident,texto,x,y,tamanho in [\n        ("LEGENDA_X","+X (m)",3.3,-.5,(1.5,.42)),\n        ("LEGENDA_Y","+Y (m)",.9,2.8,(1.5,.42)),\n        ("AVISO","PROVISORIO | grade: 1 m",cx,p["y_max"]-1,(10,.65)),\n    ]:\n        textura(ident,texto,(41,55,65))\n        partes.append(rotulo(ident,x,y,.012,tamanho))\n    ambiente = "\\n".join(partes)\n    x,y = cfg.pontos[cfg.robo["ponto_inicial"]]\n    mundos = (("etapa2_campus.wbt", "carrinho_manual", False),\n              ("etapa2_campus_teste.wbt", "teste_campus", True),\n              ("etapa3_buscas.wbt", "carrinho_buscas", True),\n              ("etapa3_buscas_teste.wbt", "teste_buscas", True),\n              ("etapa4_navegacao_teste.wbt", "teste_navegacao", True),\n              ("campus_interiores_teste.wbt", "teste_interiores", True))\n    for nome, controlador, supervisor in mundos:\n        # Supervisor: interface didática e localização ideal da navegação.\n        robo = f\'\'\'DEF ROBO Carrinho {{\n  translation {vetor((x,y,.06*cfg.robo[\'escala\']+.001))}\n  escala {cfg.robo[\'escala\']}\n  rotation 0 0 1 {cfg.robo[\'angulo\']:.10g}\n  controller "{controlador}"\n  supervisor {\'TRUE\' if supervisor else \'FALSE\'}\n}}\n\'\'\'\n        titulo = (\'IESB - TESTE AUTOMATICO: \' + controlador if controlador.startswith(\'teste_\')\n                  else \'IESB - campus ampliado - \' + (\'INTERATIVO: buscas e navegacao\' if supervisor else \'controle manual\'))\n        cena = ambiente.replace(\'title "IESB - campus ampliado - destinos internos"\', f\'title "{titulo}"\')\n        (RAIZ/"worlds"/nome).write_text(cena+robo,encoding="utf-8")\n    print(f"Gerados {len(mundos)} mundos, {len(cfg.pontos)} pontos e {len(caixas)} paredes/blocos.")\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--validar", action="store_true", help="Valida sem gerar arquivos")\n    args = parser.parse_args()\n    if args.validar:\n        validar()\n        print("Configuração válida.")\n    else:\n        gerar()\n', 'grade_navegacao.py': '"""Grade de estados do campus: geometria conhecida, sem sensores ou motores.\n\nCada célula descreve uma posição de referência (seu centro). Livre não significa\nque todo o quadrado está vazio: significa que o carrinho cabe no centro. Arestas\ntambém são verificadas, usando obstáculos expandidos por uma margem conservadora.\n"""\n\nimport math\nfrom dataclasses import dataclass\n\nimport campus_config as cfg\nfrom gerar_campus import validar, restricoes_circulacao\n\n\n@dataclass(frozen=True)\nclass Celula:\n    id: str\n    coluna: int\n    linha: int\n    centro: tuple\n    bloqueios: tuple\n\n    @property\n    def livre(self):\n        return not self.bloqueios\n\n\ndef segmento_intercepta_retangulo(a, b, limites):\n    """Recorte paramétrico exato: contato com a borda também conta como colisão."""\n    entrada, saida = 0.0, 1.0\n    for eixo, (menor, maior) in enumerate((limites[:2], limites[2:])):\n        delta = b[eixo] - a[eixo]\n        if abs(delta) < 1e-12:\n            if a[eixo] < menor or a[eixo] > maior:\n                return False\n        else:\n            t1, t2 = sorted(((menor-a[eixo])/delta, (maior-a[eixo])/delta))\n            entrada, saida = max(entrada, t1), min(saida, t2)\n            if entrada > saida:\n                return False\n    return True\n\n\nclass Grade:\n    def __init__(self, config=cfg):\n        self.config = config\n        self.caixas_fisicas = validar(config)\n        self.caixas = self.caixas_fisicas + restricoes_circulacao(config)\n        self.passo = config.grade["passo"]\n        if not math.isfinite(self.passo) or self.passo <= 0:\n            raise ValueError("Passo da grade deve ser positivo e finito")\n        self.raio = math.hypot(.12, .1125)*config.robo["escala"] + config.folga_robo\n        self.piso = config.piso\n        self.retangulos = []\n        for caixa in self.caixas:\n            x, y = caixa["centro"]\n            w, d, _ = caixa["tamanho"]\n            self.retangulos.append((caixa["id"], (x-w/2-self.raio, x+w/2+self.raio,\n                                                     y-d/2-self.raio, y+d/2+self.raio)))\n        nx = (self.piso["x_max"]-self.piso["x_min"])/self.passo\n        ny = (self.piso["y_max"]-self.piso["y_min"])/self.passo\n        if not (math.isclose(nx, round(nx)) and math.isclose(ny, round(ny))):\n            raise ValueError("O piso deve conter um número inteiro de células")\n        self.colunas, self.linhas = round(nx), round(ny)\n        self.celulas = {}\n        for linha in range(self.linhas):\n            for coluna in range(self.colunas):\n                centro = (self.piso["x_min"]+(coluna+.5)*self.passo,\n                          self.piso["y_min"]+(linha+.5)*self.passo)\n                ident = self.id_celula(coluna, linha)\n                self.celulas[ident] = Celula(ident, coluna, linha, centro,\n                                            tuple(self.bloqueios_segmento(centro, centro)))\n        self.grafo = {c.id: {} for c in self.celulas.values() if c.livre}\n        # Ordem fixa: +X, +Y, -X, -Y. A pilha da profundidade retira o último.\n        for ident in self.grafo:\n            c = self.celulas[ident]\n            for dc, dl in ((1, 0), (0, 1), (-1, 0), (0, -1)):\n                outro = self.id_celula(c.coluna+dc, c.linha+dl)\n                if outro in self.grafo:\n                    destino = self.celulas[outro].centro\n                    if self.segmento_livre(c.centro, destino):\n                        self.grafo[ident][outro] = math.dist(c.centro, destino)\n        self.componentes = {}\n        for ident in self.grafo:\n            if ident in self.componentes:\n                continue\n            numero = len(set(self.componentes.values()))\n            pendentes = [ident]\n            self.componentes[ident] = numero\n            while pendentes:\n                for vizinho in self.grafo[pendentes.pop()]:\n                    if vizinho not in self.componentes:\n                        self.componentes[vizinho] = numero\n                        pendentes.append(vizinho)\n        self.associacoes = {ident: self.associar(ponto) for ident, ponto in config.pontos.items()}\n\n    @staticmethod\n    def id_celula(coluna, linha):\n        return f"CEL_C{coluna:02d}_L{linha:02d}"\n\n    def mundo_para_celula(self, x, y):\n        """Célula que contém o ponto; limites superiores do piso são exclusivos."""\n        p = self.piso\n        if not (p["x_min"] <= x < p["x_max"] and p["y_min"] <= y < p["y_max"]):\n            raise ValueError("Coordenada fora da grade")\n        return self.id_celula(math.floor((x-p["x_min"])/self.passo),\n                              math.floor((y-p["y_min"])/self.passo))\n\n    def bloqueios_segmento(self, a, b):\n        p, r = self.piso, self.raio\n        if any(not (p["x_min"]+r <= x <= p["x_max"]-r and\n                    p["y_min"]+r <= y <= p["y_max"]-r) for x, y in (a, b)):\n            yield "BORDA_PISO"\n        for ident, limites in self.retangulos:\n            if segmento_intercepta_retangulo(a, b, limites):\n                yield ident\n\n    def segmento_livre(self, a, b):\n        return not any(self.bloqueios_segmento(a, b))\n\n    def associar(self, ponto):\n        """Centro livre mais próximo com conector seguro; nunca atravessa paredes.\n\n        O conector não é uma aresta da grade. Seu comprimento é informado à parte.\n        Empates: coluna e depois linha. Não escolhe componente para ocultar falhas.\n        """\n        limite = self.config.grade["distancia_max_associacao"]\n        if not math.isfinite(limite) or limite < 0:\n            raise ValueError("Limite de associação deve ser finito e não negativo")\n        # Distâncias iguais a 1 nanômetro usam o ID como desempate. Isso evita\n        # trocar de célula por ruído de ponto flutuante do simulador (ex.: 1e-15 m).\n        candidatos = sorted(((math.dist(ponto, c.centro), c.id) for c in self.celulas.values()\n                             if c.livre and math.dist(ponto, c.centro) <= limite),\n                            key=lambda item: (round(item[0], 9), item[1]))\n        for distancia, ident in candidatos:\n            if self.segmento_livre(ponto, self.celulas[ident].centro):\n                return {"celula": ident, "distancia_m": distancia,\n                        "componente": self.componentes[ident]}\n        return {"celula": None, "distancia_m": None, "componente": None}\n\n    def heuristica(self, destino, grafo=None):\n        """Manhattan em metros: admissível e consistente para quatro vizinhos."""\n        x, y = self.celulas[destino].centro\n        return {ident: abs(c.centro[0]-x)+abs(c.centro[1]-y)\n                for ident in (self.grafo if grafo is None else grafo)\n                for c in (self.celulas[ident],)}\n\n    def recorte(self, limites):\n        """Subgrafo induzido: borda do recorte limita a busca, não cria parede física."""\n        xa, xb, ya, yb = limites\n        ids = {n for n in self.grafo if xa <= self.celulas[n].centro[0] < xb\n               and ya <= self.celulas[n].centro[1] < yb}\n        return {n: {v: c for v, c in vs.items() if v in ids}\n                for n, vs in self.grafo.items() if n in ids}\n', 'buscas.py': '"""Buscas adaptadas do Laboratório 2 do Prof. Rodrigo Gonçalves Pinto.\r\n\r\nGerado por adaptar_laborator\r\nio2.py. O anexo original permanece intacto.\r\nMapa: dicionário de dicionários, com custos finitos e não negativos.\r\nA* mantém o conjunto fechado didático: requer heurística CONSISTENTE.\r\nManhattan em grade de quatro vizinhos satisfaz essa condição.\r\nObjetivo testado ao retirar da fronteira; objetivo não conta como expandido.\r\nRetorno: caminho, custo, quantidade de expansões, ordem de expansão.\r\n"""\r\nfrom collections import deque\r\nimport heapq\r\nimport itertools\r\nimport math\r\n\r\n\r\ndef validar_grafo(mapa):\r\n    """Validação anterior à medição; aceita grafos dirigidos ou não dirigidos."""\r\n    for origem, vizinhos in mapa.items():\r\n        for destino, custo in vizinhos.items():\r\n            if destino not in mapa or not math.isfinite(custo) or custo < 0:\r\n                raise ValueError(f"Aresta inválida: {origem} -> {destino}")\r\n\r\n\r\ndef conferir_extremos(mapa, inicio, objetivo):\r\n    """Distingue estado inválido de estado válido sem caminho até o destino."""\r\n    if inicio not in mapa or objetivo not in mapa:\r\n        raise ValueError("Origem ou destino ausente do grafo navegável")\r\n\r\n\r\n# Origem: Laboratório 2, célula 12 (índice começando em zero).\r\nclass No:\r\n    """Um nó da árvore de busca: estado + memória de como chegamos até ele."""\r\n\r\n    __slots__ = ("estado", "pai", "acao", "g")\r\n\r\n    def __init__(self, estado, pai=None, acao=None, g=0):\r\n        self.estado = estado    # a cidade\r\n        self.pai = pai          # o nó anterior (None se for a raiz)\r\n        self.acao = acao        # a ação aplicada no pai\r\n        self.g = g              # custo acumulado desde o início\r\n\r\n    def __repr__(self):\r\n        # __repr__ define o que aparece quando imprimimos o objeto.\r\n        # Sem ele, o Python mostraria algo como <__main__.No object at 0x7f...>.\r\n        return f"No({self.estado}, g={self.g})"\r\n\r\n    def __lt__(self, outro):\r\n        # __lt__ define o operador "<". O heapq (fila de prioridade) precisa\r\n        # dele para desempatar quando duas prioridades forem iguais.\r\n        return self.g < outro.g\r\n\r\n\r\n# Origem: Laboratório 2, célula 14 (índice começando em zero).\r\ndef reconstruir_caminho(no):\r\n    """Devolve a lista de estados da raiz até o nó recebido.\r\n\r\n    Parâmetro:\r\n        no (No): o nó final, normalmente o que satisfez o teste de objetivo.\r\n\r\n    Devolve:\r\n        list: as cidades em ordem, do início até o nó.\r\n    """\r\n    caminho = []                    # lista vazia para acumular os estados\r\n\r\n    while no is not None:           # "is not None" para quando chegar na raiz\r\n        caminho.append(no.estado)   # anota o estado do nó atual\r\n        no = no.pai                 # sobe um nível na cadeia\r\n\r\n    # Neste ponto a lista está de TRÁS PARA FRENTE: [Pitesti, ..., Arad].\r\n    # A fatia [::-1] percorre a lista de ré e devolve uma cópia invertida.\r\n    return caminho[::-1]\r\n\r\n\r\n# Origem: Laboratório 2, célula 8 (índice começando em zero).\r\ndef custo_do_caminho(mapa, caminho):\r\n    """Soma os custos de passo ao longo de uma lista de estados.\r\n\r\n    Parâmetro:\r\n        caminho (list): lista de estados em ordem, ex. ["Arad", "Sibiu"].\r\n\r\n    Devolve:\r\n        int: o custo total do caminho, em unidades do mapa (metros no campus).\r\n    """\r\n    # TODO 1 resolvido: soma de todos os pares consecutivos.\r\n    if caminho is None:\r\n        return None\r\n    return sum(mapa[a][b] for a, b in zip(caminho, caminho[1:]))\r\n\r\n\r\n# Origem: Laboratório 2, célula 25 (índice começando em zero).\r\ndef busca_profundidade(mapa, inicio, objetivo):\r\n    """Busca em profundidade em grafo. Não é ótima — e é esse o ponto."""\r\n    conferir_extremos(mapa, inicio, objetivo)\r\n    pilha = [No(inicio)]\r\n    explorados = set()\r\n    ordem = []\r\n\r\n    while pilha:\r\n        no = pilha.pop()                     # retira o mais recente: pilha\r\n\r\n        if no.estado == objetivo:\r\n            return reconstruir_caminho(no), no.g, len(ordem), ordem\r\n\r\n        if no.estado in explorados:\r\n            continue\r\n\r\n        explorados.add(no.estado)\r\n        ordem.append(no.estado)\r\n\r\n        for vizinho, custo in mapa[no.estado].items():\r\n            if vizinho not in explorados:\r\n                pilha.append(No(vizinho, pai=no, acao=vizinho, g=no.g + custo))\r\n\r\n    return None, None, len(ordem), ordem\r\n\r\n\r\n# Origem: Laboratório 2, célula 17 (índice começando em zero).\r\ndef busca_largura(mapa, inicio, objetivo):\r\n    """Busca em largura em grafo.\r\n\r\n    Parâmetros:\r\n        inicio (str): estado de partida.\r\n        objetivo (str): estado de destino.\r\n\r\n    Devolve:\r\n        tuple: (caminho, custo, nós expandidos, ordem de expansão).\r\n    """\r\n    conferir_extremos(mapa, inicio, objetivo)\r\n    fronteira = deque([No(inicio)])   # a fronteira começa com o nó inicial\r\n    explorados = set()                # conjunto: consulta imediata\r\n    na_fronteira = {inicio}           # espelho da fronteira, para consulta rápida\r\n    ordem = []                        # registra a ordem das expansões\r\n\r\n    while fronteira:                  # enquanto houver nós para examinar\r\n        no = fronteira.popleft()      # retira o mais antigo: regra da fila\r\n        na_fronteira.discard(no.estado)\r\n\r\n        # Teste de objetivo NA REMOÇÃO, conforme a convenção da disciplina.\r\n        if no.estado == objetivo:\r\n            return reconstruir_caminho(no), no.g, len(ordem), ordem\r\n\r\n        explorados.add(no.estado)     # marca como já expandido\r\n        ordem.append(no.estado)\r\n\r\n        for vizinho, custo in mapa[no.estado].items():\r\n            if vizinho in explorados or vizinho in na_fronteira:\r\n                continue              # já visto: ignora e vai para o próximo\r\n\r\n            filho = No(vizinho, pai=no, acao=vizinho, g=no.g + custo)\r\n\r\n            # Lacuna do laboratório resolvida: inserção na fronteira.\r\n            fronteira.append(filho)\r\n\r\n            na_fronteira.add(vizinho)\r\n\r\n    return None, None, len(ordem), ordem\r\n\r\n\r\n# Origem: Laboratório 2, célula 21 (índice começando em zero).\r\ndef busca_custo_uniforme(mapa, inicio, objetivo):\r\n    """Expande sempre o nó de menor custo acumulado g(n).\r\n\r\n    Parâmetros:\r\n        inicio (str): estado de partida.\r\n        objetivo (str): estado de destino.\r\n\r\n    Devolve:\r\n        tuple: (caminho, custo, nós expandidos, ordem de expansão).\r\n    """\r\n    conferir_extremos(mapa, inicio, objetivo)\r\n    contador = itertools.count()      # gera 0, 1, 2, 3... para desempate\r\n    fronteira = []\r\n    heapq.heappush(fronteira, (0, next(contador), No(inicio)))\r\n    explorados = set()\r\n    ordem = []\r\n\r\n    while fronteira:\r\n        # heappop devolve a tupla inteira; "_" marca o que não vamos usar.\r\n        _, _, no = heapq.heappop(fronteira)\r\n\r\n        if no.estado == objetivo:\r\n            return reconstruir_caminho(no), no.g, len(ordem), ordem\r\n\r\n        # Um mesmo estado pode ter entrado várias vezes com custos diferentes.\r\n        # Se já expandimos este estado, esta cópia é obsoleta: descartamos.\r\n        if no.estado in explorados:\r\n            continue\r\n\r\n        explorados.add(no.estado)\r\n        ordem.append(no.estado)\r\n\r\n        for vizinho, custo in mapa[no.estado].items():\r\n            if vizinho in explorados:\r\n                continue\r\n\r\n            filho = No(vizinho, pai=no, acao=vizinho, g=no.g + custo)\r\n\r\n            # Lacuna do laboratório resolvida: inserção na fronteira.\r\n            heapq.heappush(fronteira, (filho.g, next(contador), filho))\r\n\r\n    return None, None, len(ordem), ordem\r\n\r\n\r\n# Origem: Laboratório 2, célula 31 (índice começando em zero).\r\ndef busca_gulosa(mapa, inicio, objetivo, h):\r\n    """Ordena a fronteira apenas pela estimativa h(n)."""\r\n    conferir_extremos(mapa, inicio, objetivo)\r\n    contador = itertools.count()\r\n    fronteira = []\r\n    heapq.heappush(fronteira, (h[inicio], next(contador), No(inicio)))\r\n    explorados = set()\r\n    ordem = []\r\n\r\n    while fronteira:\r\n        _, _, no = heapq.heappop(fronteira)\r\n\r\n        if no.estado == objetivo:\r\n            return reconstruir_caminho(no), no.g, len(ordem), ordem\r\n\r\n        if no.estado in explorados:\r\n            continue\r\n\r\n        explorados.add(no.estado)\r\n        ordem.append(no.estado)\r\n\r\n        for vizinho, custo in mapa[no.estado].items():\r\n            if vizinho in explorados:\r\n                continue\r\n            filho = No(vizinho, pai=no, acao=vizinho, g=no.g + custo)\r\n            # A prioridade é SÓ a heurística: o passado é ignorado.\r\n            heapq.heappush(fronteira, (h[vizinho], next(contador), filho))\r\n\r\n    return None, None, len(ordem), ordem\r\n\r\n\r\n# Origem: Laboratório 2, célula 35 (índice começando em zero).\r\ndef a_estrela(mapa, inicio, objetivo, h):\r\n    """Ordena a fronteira por f(n) = g(n) + h(n).\r\n\r\n    Parâmetros:\r\n        inicio (str): estado de partida.\r\n        objetivo (str): estado de destino.\r\n        h (dict): a função heurística, como dicionário estado -> estimativa.\r\n\r\n    Devolve:\r\n        tuple: (caminho, custo, nós expandidos, ordem de expansão).\r\n    """\r\n    conferir_extremos(mapa, inicio, objetivo)\r\n    contador = itertools.count()\r\n    fronteira = []\r\n    heapq.heappush(fronteira, (h[inicio], next(contador), No(inicio)))\r\n    explorados = set()\r\n    ordem = []\r\n\r\n    while fronteira:\r\n        _, _, no = heapq.heappop(fronteira)\r\n\r\n        if no.estado == objetivo:\r\n            return reconstruir_caminho(no), no.g, len(ordem), ordem\r\n\r\n        if no.estado in explorados:\r\n            continue\r\n\r\n        explorados.add(no.estado)\r\n        ordem.append(no.estado)\r\n\r\n        for vizinho, custo in mapa[no.estado].items():\r\n            if vizinho in explorados:\r\n                continue\r\n\r\n            filho = No(vizinho, pai=no, acao=vizinho, g=no.g + custo)\r\n\r\n            # Lacuna do laboratório resolvida: inserção na fronteira.\r\n            heapq.heappush(fronteira, (filho.g + h[vizinho], next(contador), filho))\r\n\r\n    return None, None, len(ordem), ordem\r\n', 'planejamento.py': '"""Interface de planejamento compartilhada pelo Webots e pelo Colab (Python puro)."""\nimport math\nimport re\nimport unicodedata\nfrom time import perf_counter_ns\n\nimport buscas\n\nALGORITMOS = (\n    ("Profundidade", buscas.busca_profundidade, False),\n    ("Largura", buscas.busca_largura, False),\n    ("Custo uniforme", buscas.busca_custo_uniforme, False),\n    ("Gulosa", buscas.busca_gulosa, True),\n    ("A*", buscas.a_estrela, True),\n)\n\n\ndef normalizar_nome(texto):\n    return \'\'.join(c for c in unicodedata.normalize(\'NFD\', texto.upper()) if c.isalnum())\n\n\ndef alvo_do_clique(grade, x, y, selecionar_setor=True):\n    """Destino: clique no interior/nome do bloco escolhe seu centro cadastrado.\n\n    Fora dos setores (ou ao definir origem), preserva a seleção de célula.\n    A faixa ocupada pelas paredes não é interpretada como clique no interior.\n    """\n    if not all(math.isfinite(v) for v in (x, y)):\n        raise ValueError(\'Clique sem posição válida no piso.\')\n    if selecionar_setor:\n        t = grade.config.paredes_padrao[\'espessura\']/2\n        destinos = getattr(grade.config, \'destinos_setores\', {})\n        for setor in grade.config.setores:\n            cx, cy = setor[\'centro\']\n            w, d = setor[\'tamanho\']\n            if abs(x-cx) < w/2-t and abs(y-cy) < d/2-t and setor[\'id\'] in destinos:\n                return destinos[setor[\'id\']]\n    return grade.mundo_para_celula(x, y)\n\n\ndef resolver_alvo(grade, texto):\n    """Aceita H01, Bloco H, H, nome de setor, DEF ou célula; valida sem aproximar parede."""\n    valor = texto.strip().upper().replace(" ", "")\n    aliases = {normalizar_nome(nome): ident for ident, nome in grade.config.nomes.items()}\n    for setor, ident in getattr(grade.config, \'destinos_setores\', {}).items():\n        aliases[normalizar_nome(setor)] = ident\n        if setor.startswith(\'BLOCO_\'):\n            aliases[setor[-1]] = ident\n    if valor not in grade.config.pontos:\n        valor = aliases.get(normalizar_nome(texto), valor)\n    if valor in grade.config.pontos:\n        ponto = grade.config.pontos[valor]\n        ident = grade.associacoes[valor]["celula"]\n        if ident is None:\n            raise ValueError(f"Local {valor} sem associação segura")\n        return ident, ponto\n    match = re.fullmatch(r"(?:CEL_)?C(\\d+)_?L(\\d+)", valor) or re.fullmatch(r"(\\d+),(\\d+)", valor)\n    if not match:\n        raise ValueError("Use Bloco H, H01, Biblioteca ou uma célula como C29L25")\n    ident = grade.id_celula(*map(int, match.groups()))\n    if ident not in grade.celulas:\n        raise ValueError("Célula fora da grade")\n    if ident not in grade.grafo:\n        raise ValueError(f"{ident}: centro bloqueado para o carrinho")\n    return ident, grade.celulas[ident].centro\n\n\ndef planejar(grade, posicao_inicial, destino, algoritmo=4):\n    """Associa a posição atual e mede somente a busca; não movimenta motores.\n\n    Destino é um texto aceito por resolver_alvo. Célula livre desconectada é um\n    resultado válido sem solução; célula bloqueada ou posição insegura gera erro.\n    """\n    if not 0 <= algoritmo < len(ALGORITMOS):\n        raise ValueError("Algoritmo deve ser um índice entre 0 e 4")\n    if len(posicao_inicial) != 2 or not all(math.isfinite(v) for v in posicao_inicial):\n        raise ValueError("Posição inicial inválida")\n    # A associação já verifica o segmento desde a posição contínua do robô.\n    origem = grade.associar(posicao_inicial)\n    if origem["celula"] is None:\n        raise ValueError("Posição atual sem centro acessível; defina outra origem com O")\n    z, ponto_final = resolver_alvo(grade, destino)\n    nome, funcao, usa_h = ALGORITMOS[algoritmo]\n    a = origem["celula"]\n    args = (grade.grafo, a, z) + ((grade.heuristica(z),) if usa_h else ())\n    inicio = perf_counter_ns()\n    caminho, custo, expandidos, ordem = funcao(*args)\n    tempo_ms = (perf_counter_ns()-inicio)/1e6\n    assert custo == buscas.custo_do_caminho(grade.grafo, caminho)\n    conectores = origem["distancia_m"]+math.dist(ponto_final, grade.celulas[z].centro)\n    if tuple(posicao_inicial) == tuple(ponto_final):\n        conectores = 0.0\n    return {"algoritmo": nome, "origem": a, "destino": z,\n            "posicao_inicial": tuple(posicao_inicial), "ponto_final": tuple(ponto_final),\n            "caminho": caminho, "custo_m": custo, "expandidos": expandidos,\n            "ordem": ordem, "tempo_ms": tempo_ms, "conectores_m": conectores,\n            "percurso_total_m": None if custo is None else custo+conectores}\n\n\ndef planejar_multiparadas(grade, posicao_inicial, lista_destinos, algoritmo=4):\n    """Planeja percurso sequencial passando por múltiplos destinos sucessivos.\n\n    Exemplo: lista_destinos=[\'CANTINA\', \'H01\', \'EXT01\']\n    Devolve resultado unificado compatível com a navegação do carrinho.\n    """\n    if not lista_destinos:\n        raise ValueError("Lista de destinos não pode ser vazia")\n    pos_atual = tuple(posicao_inicial)\n    caminho_total = []\n    ordem_total = []\n    custo_total = 0.0\n    expandidos_total = 0\n    tempo_total_ms = 0.0\n    conectores_total = 0.0\n    trechos = []\n\n    for dest in lista_destinos:\n        r = planejar(grade, pos_atual, dest, algoritmo)\n        if r["caminho"] is None:\n            return {"algoritmo": r["algoritmo"], "origem": r["origem"], "destino": dest,\n                    "posicao_inicial": tuple(posicao_inicial), "ponto_final": r["ponto_final"],\n                    "caminho": None, "custo_m": None, "expandidos": expandidos_total + r["expandidos"],\n                    "ordem": ordem_total + r["ordem"], "tempo_ms": tempo_total_ms + r["tempo_ms"],\n                    "conectores_m": None, "percurso_total_m": None, "trechos": trechos}\n        trechos.append(r)\n        if not caminho_total:\n            caminho_total.extend(r["caminho"])\n        else:\n            if r["caminho"] and caminho_total[-1] == r["caminho"][0]:\n                caminho_total.extend(r["caminho"][1:])\n            else:\n                caminho_total.extend(r["caminho"])\n        ordem_total.extend(r["ordem"])\n        custo_total += r["custo_m"]\n        expandidos_total += r["expandidos"]\n        tempo_total_ms += r["tempo_ms"]\n        conectores_total += r["conectores_m"]\n        pos_atual = r["ponto_final"]\n\n    ponto_final = trechos[-1]["ponto_final"]\n    return {\n        "algoritmo": trechos[0]["algoritmo"],\n        "origem": trechos[0]["origem"],\n        "destino": trechos[-1]["destino"],\n        "destinos_intermediarios": list(lista_destinos),\n        "posicao_inicial": tuple(posicao_inicial),\n        "ponto_final": ponto_final,\n        "caminho": caminho_total,\n        "custo_m": custo_total,\n        "expandidos": expandidos_total,\n        "ordem": ordem_total,\n        "tempo_ms": tempo_total_ms,\n        "conectores_m": conectores_total,\n        "percurso_total_m": custo_total + conectores_total,\n        "trechos": trechos,\n    }\n'}

pasta_do_mapa = Path(tempfile.mkdtemp(prefix="iesb_mapa_antigo_"))
for nome, conteudo in arquivos_do_mapa.items():
    (pasta_do_mapa / nome).write_text(conteudo, encoding="utf-8")
    # Se a célula for repetida, a próxima importação usará esta mesma cópia.
    sys.modules.pop(Path(nome).stem, None)

sys.path.insert(0, str(pasta_do_mapa))
print("Mapa antigo disponível. Continue para a seção 1.")

## 1. Preparar o mapa

In [ ]:
# import permite usar funções que já estão em outros arquivos do projeto.
# json transforma dicionários Python em texto organizado para o modelo ler.
import json

from grade_navegacao import Grade
from planejamento import normalizar_nome, planejar

# A grade divide o mapa em quadrados e registra as ligações livres entre eles.
# Construímos uma única vez: todas as consultas abaixo usam o mesmo mapa.
# É o campus provisório de 50 x 30 m, chamado aqui de "mapa antigo".
grade = Grade()

In [ ]:
# Veja quais nomes e IDs podem ser usados nas consultas.
# O f antes das aspas permite colocar variáveis dentro de {chaves}.
for identificador, nome in grade.config.nomes.items():
    print(f"{identificador}: {nome}")

## 2. Encontrar um local pelo nome

In [ ]:
def encontrar_local(texto):
    """Recebe um nome ou ID cadastrado e devolve seu ID, como BB01.

    Exemplos aceitos: 'Biblioteca', 'biblioteca', 'BB01', 'Bloco H'.
    Ainda não interpretamos frases como 'quero ir até a biblioteca'.
    """
    # isinstance verifica o tipo. Precisamos de texto, e não de número ou lista.
    # strip remove espaços nas pontas; texto vazio não identifica um local.
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("Digite o nome ou o ID de um local do mapa.")

    # A função já existente remove acentos e diferenças de maiúsculas/espaços.
    # Assim, 'entrada iesb' e 'Entrada IESB' podem identificar o mesmo lugar.
    nome_procurado = normalizar_nome(texto)

    # nomes é um dicionário: cada ID está associado a um nome legível.
    # items() permite percorrer os pares, por exemplo ('BB01', 'Biblioteca').
    for identificador, nome in grade.config.nomes.items():
        if nome_procurado in (normalizar_nome(identificador), normalizar_nome(nome)):
            return identificador

    # Não encontramos? Interrompemos a consulta em vez de adivinhar um destino.
    # O notebook mostra a lista de nomes válidos antes do primeiro exemplo.
    raise ValueError(f"Local não cadastrado: {texto!r}. Consulte a lista de locais.")

## 3. Calcular a rota com o A*

In [ ]:
def calcular_rota(origem, destino):
    """Calcula uma rota entre dois locais cadastrados, sem movimentar o robô.

    Entrada: dois textos, como 'Entrada IESB' e 'Biblioteca'.
    Saída: um dicionário com os nomes, o caminho e as distâncias em metros.
    Local inválido gera ValueError. Local válido sem caminho retorna 'sem_rota'.
    """
    # Primeiro conferimos os DOIS locais. O modelo de linguagem não faz isso.
    id_origem = encontrar_local(origem)
    id_destino = encontrar_local(destino)

    # pontos guarda as coordenadas (x, y) de cada marcador no mapa.
    # O planejador precisa da posição da origem para encontrar a célula inicial.
    posicao_inicial = grade.config.pontos[id_origem]

    # Reaproveitamos a mesma função usada pelo projeto no Webots.
    # A lista de algoritmos começa no índice 0; o A* está no índice 4.
    # A* combina o custo já percorrido com uma estimativa até o destino.
    # A grade fornece as conexões livres e a estimativa de distância Manhattan.
    resultado = planejar(grade, posicao_inicial, id_destino, algoritmo=4)

    # None significa ausência de caminho. É diferente de uma rota de custo zero,
    # que acontece quando a origem e o destino são o mesmo lugar.
    if resultado["caminho"] is None:
        status = "sem_rota"
        conectores = None
    else:
        status = "ok"
        conectores = resultado["conectores_m"]

    # Um dicionário guarda valores com nomes (as "chaves" entre aspas).
    # Usamos nomes oficiais, mesmo quando a pessoa digitou IDs ou minúsculas.
    # O caminho é uma lista de IDs de CÉLULAS, não uma lista de prédios visitados.
    # Não arredondamos aqui: preservamos a precisão do cálculo original.
    return {
        "status": status,
        "mapa": "Campus provisório de 50 x 30 m (mapa antigo)",
        "origem": grade.config.nomes[id_origem],
        "destino": grade.config.nomes[id_destino],
        "algoritmo": resultado["algoritmo"],
        "caminho": resultado["caminho"],
        # Distância percorrida pelas conexões entre centros das células.
        "distancia_grade_m": resultado["custo_m"],
        # Pequenos trechos entre os marcadores dos locais e os centros da grade.
        "conectores_m": conectores,
        # Soma da distância na grade com os conectores: use esta no resumo.
        "distancia_total_m": resultado["percurso_total_m"],
    }

## 4. Preparar os dados que o Qwen vai explicar

In [ ]:
def montar_mensagens(rota):
    """Transforma a saída de calcular_rota em uma conversa para o Qwen.

    Aqui só preparamos texto. A geração da resposta ocorre depois no notebook.
    Passe o dicionário calculado pelo Python, não um resultado criado pelo modelo.
    """
    # Se não existe caminho, mostramos o aviso diretamente no notebook.
    # Não precisamos pedir para um modelo inventar uma explicação de percurso.
    if rota["status"] != "ok":
        raise ValueError("Não há rota para explicar neste mapa.")

    # system contém as orientações de resposta; user contém os dados calculados.
    # Um prompt orienta o modelo, mas não garante que ele obedeça perfeitamente.
    # Por isso, o notebook também mostra o resultado original para conferência.
    orientacao = (
        "Explique uma rota já calculada por Python. Responda em português, "
        "em até quatro frases. Use somente os dados fornecidos. Informe origem, "
        "destino, algoritmo e distancia_total_m em metros, com duas casas decimais. "
        "Identifique o mapa como provisório. Os itens de caminho são células, "
        "não nomes de prédios. Não invente locais intermediários, direções de "
        "curvas, tempos de viagem, obstáculos, acessibilidade real ou bateria. "
        "Não recalcule nem altere a rota. Não diga que o carrinho já se moveu. "
        "Não confunda distancia_grade_m com distancia_total_m."
    )

    # ensure_ascii=False mantém os acentos legíveis; indent=2 organiza as linhas.
    # JSON é apenas a forma de enviar os dados. Não estamos pedindo saída JSON
    # ao modelo nesta etapa (isso será trabalhado no Laboratório 8).
    dados_em_texto = json.dumps(rota, ensure_ascii=False, indent=2)

    return [
        {"role": "system", "content": orientacao},
        {"role": "user", "content": "Explique este resultado calculado:\n" + dados_em_texto},
    ]

## 5. Experimentar uma rota sem usar o modelo

Primeiro conferimos o cálculo sozinho. As células são os estados da busca;
não representam prédios. A distância total inclui os pequenos conectores
entre os marcadores e os centros das células. **Metros não são segundos.**
O tempo de execução do algoritmo também não seria o tempo de viagem.

In [ ]:
# Troque estes dois textos para experimentar outros locais cadastrados.
origem = "Entrada IESB"
destino = "Biblioteca"

# Começar com None evita reutilizar uma rota antiga se a nova consulta falhar.
rota = None
try:
    rota = calcular_rota(origem, destino)
except ValueError as erro:
    # try/except permite mostrar uma mensagem compreensível em caso de erro.
    print("Não foi possível calcular:", erro)

if rota is not None:
    if rota["status"] == "sem_rota":
        print("Não existe caminho entre esses locais neste mapa.")
    else:
        print("Resultado calculado pelo Python (referência para conferir o Qwen):")
        print("Origem:", rota["origem"])
        print("Destino:", rota["destino"])
        print("Algoritmo:", rota["algoritmo"])
        # :.2f mostra duas casas decimais sem alterar o valor guardado.
        print(f"Na grade: {rota['distancia_grade_m']:.2f} m")
        print(f"Conectores: {rota['conectores_m']:.2f} m")
        print(f"Total: {rota['distancia_total_m']:.2f} m")
        # join une os IDs com uma seta, mostrando a ordem do caminho.
        print("Células:", " → ".join(rota["caminho"]))

## 6. Entender os casos especiais

Origem igual ao destino tem distância zero. Um nome inexistente gera erro:
isso não significa que o caminho está bloqueado. São situações diferentes.
A ausência de caminho entre locais válidos também é tratada pela função,
mas todos os locais cadastrados estão conectados no mapa antigo atual.

In [ ]:
# Este exemplo não muda a variável rota usada na consulta principal.
mesma_origem = calcular_rota("Biblioteca", "BB01")
print("Mesmo lugar:", mesma_origem["distancia_total_m"], "m")

try:
    calcular_rota("Entrada IESB", "Local que não existe")
except ValueError as erro:
    print("Erro esperado:", erro)

## 7. Ver exatamente o que será enviado ao Qwen

Uma conversa é uma lista de mensagens. Cada mensagem tem `role` (quem fala)
e `content` (o texto). A mensagem `system` orienta a resposta; a mensagem
`user` leva os dados calculados. O modelo vai explicar esses dados.

In [ ]:
mensagens = None
if rota is not None and rota["status"] == "ok":
    mensagens = montar_mensagens(rota)
    for mensagem in mensagens:
        print("\nPapel:", mensagem["role"])
        print(mensagem["content"])
else:
    print("Corrija a consulta na seção 5 antes de pedir uma explicação.")

## 8. Instalar e carregar o Qwen

Usamos `Qwen/Qwen2.5-1.5B-Instruct`, o modelo do Laboratório 7.
`transformers` fornece o carregamento e a geração; `torch` executa as
operações numéricas e já vem no Colab. Não é necessário colocar token
pessoal no notebook para este modelo público.

In [ ]:
# %pip instala no mesmo ambiente Python que executa este notebook.
# A faixa mantém a API da série 4 usada nesta implementação.
%pip install -q "transformers>=4.45,<5"

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

nome_modelo = "Qwen/Qwen2.5-1.5B-Instruct"

# CUDA permite usar uma GPU compatível. Sem ela, usamos o processador (CPU).
if torch.cuda.is_available():
    dispositivo = "cuda"
    tipo_numerico = torch.float16  # Menor consumo de memória na GPU.
else:
    dispositivo = "cpu"
    tipo_numerico = torch.float32  # Opção simples para executar na CPU.

print("Dispositivo escolhido:", dispositivo)

# O tokenizador transforma texto em tokens (números que representam texto).
tokenizador = AutoTokenizer.from_pretrained(nome_modelo)

# O modelo contém os pesos já treinados. Aqui NÃO estamos treinando uma IA.
# .to coloca os pesos no dispositivo escolhido; eval ativa o modo de uso.
qwen = AutoModelForCausalLM.from_pretrained(
    nome_modelo,
    torch_dtype=tipo_numerico,
).to(dispositivo)
qwen.eval()
print("Qwen carregado. Agora podemos gerar a explicação.")

## 9. Gerar somente a resposta nova

A estrutura abaixo acompanha o Laboratório 7: preparar a conversa,
gerar tokens e converter a resposta de volta em texto. Não há chamada
automática de ferramentas nesta etapa; nós chamamos o A* na seção 5.

In [ ]:
def conversar(mensagens, max_tokens=180):
    """Recebe a conversa e devolve o texto novo produzido pelo Qwen."""
    # O template organiza os papéis no formato que este modelo aprendeu.
    # add_generation_prompt marca onde começa a resposta do assistente.
    # return_tensors="pt" cria os tensores do PyTorch.
    entrada = tokenizador.apply_chat_template(
        mensagens,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(dispositivo)

    # Sem treinamento, não precisamos guardar cálculos de gradientes.
    with torch.no_grad():
        saida = qwen.generate(
            **entrada,  # Entrega ao modelo os campos preparados acima.
            max_new_tokens=max_tokens,  # Limite de tokens novos, não de palavras.
            do_sample=False,  # Não sorteia o próximo token.
            pad_token_id=tokenizador.eos_token_id,
        )

    # generate devolve a entrada seguida da resposta. Removemos a entrada
    # para não imprimir outra vez todas as instruções e os dados da rota.
    tamanho_entrada = entrada["input_ids"].shape[-1]
    tokens_novos = saida[0][tamanho_entrada:]

    # decode converte números em texto; skip_special_tokens remove marcadores.
    resposta = tokenizador.decode(tokens_novos, skip_special_tokens=True)
    return resposta.strip()

In [ ]:
# Só geramos texto quando o cálculo anterior encontrou uma rota válida.
if mensagens is not None:
    explicacao = conversar(mensagens)
    print("Explicação produzida pelo Qwen (confira com a seção 5):")
    print(explicacao)
else:
    print("Não há rota válida para explicar. Volte à seção 5.")

## 10. Conferir e explicar ao grupo

Compare a resposta com a seção 5: os locais estão corretos? O total em metros
confere? Apareceu algum prédio intermediário, tempo ou direção não informado?
Se houver informação inventada, registre o caso e revise o prompt; não a
considere resultado do planejamento. O prompt não garante ausência de erros.

Para praticar, experimente Entrada IESB → Cantina, Biblioteca → Bloco H e
Biblioteca → Biblioteca. Execute novamente desde a seção 5; o modelo já
carregado pode ser reutilizado, sem repetir as instalações ou a seção 8.

**O que esta etapa faz:** reutiliza a busca e fornece seus dados ao modelo.
**Próximas etapas:** interpretar pedidos em linguagem natural com validação
(Lab 8) e permitir chamadas controladas de ferramentas (Lab 9). Prever tempo
de viagem por aprendizado de máquina também continua pendente.

O notebook não aciona o carrinho. A grade usa a geometria conhecida do mapa;
esta demonstração não valida localização por sensores de um carrinho físico.

Referências de API: [modelo Qwen](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct)
e [conversas no Transformers](https://huggingface.co/docs/transformers/chat_templating).